# Reading Refeyn mass photometry `.mpr` files in Python

This notebook shows how to use `mprfile` to open a Refeyn **AcquireMP** `.mpr` file, look at what's inside, and run analyses that go beyond the standard software.

**Contents**
1. Open a file and read the metadata
2. The raw movie
3. The ratiometric (contrast) movie
4. The fitted events from AcquireMP
5. Contrast histogram and the Gaussian fits stored in the file
6. Extra analyses: your own peak fitting, ladder linearity, **mass calibration**, landing kinetics, drift, and the average PSF
7. From contrast to mass: analysing a sample (automatic peaks, settings, the peak finder)
8. Calibrations: other calibrants, self-checks, saving and reusing
9. Choosing regions and the number of peaks yourself, with **overfitting statistics** (BIC, resolution, bootstrap)
10. Many files at once, and the `mpr-analyze` command
11. Export (CSV, TIFF for Fiji)
12. The **interactive ROI tool**: drag regions on the histogram

**Setup:** `conda env create -f environment.yml`, then `conda activate mpr`, then `jupyter lab`.

**Just want masses for your samples?** Use `analyze_measurements.ipynb`. This tutorial shows what's inside the files, how the analysis works, and every function for your own scripts.

**Files used:** `002_Ladder.mpr` (MassFerence P1 calibrant) and `019_252_50nM.mpr` (a sample). Change `DATA` and `SAMPLE` for your own.

**Background:** a `.mpr` file is an HDF5 file. The movie is Zstandard-compressed and stored as frame-to-frame differences. `mprfile` takes care of both, so you work with plain numpy arrays.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d

from mprfile import MPRFile, Calibration

# One colour for data, a neutral grey for fits/guides, and one accent colour for highlights
C_DATA, C_FIT, C_ACCENT = "#3b6ea5", "#555555", "#d9722e"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "image.cmap": "gray"})

DATA = Path("002_Ladder.mpr")   # <-- change to your own file

## 1. Open a file and read the metadata

`MPRFile` works as a context manager (`with MPRFile(...) as m:`), but in a notebook it's easier to keep it open and call `m.close()` at the end.

In [ ]:
m = MPRFile(DATA)
print(m.summary())

Sample, camera and instrument information are available as dictionaries:

In [ ]:
pd.concat({"sample": pd.Series(m.sample_info),
           "camera": pd.Series(m.camera),
           "instrument": pd.Series(m.instrument)}).to_frame("value")

The analysis settings AcquireMP used (the `n_avg` value there is also used by `ratiometric()` below):

In [ ]:
pd.json_normalize(m.analysis_params).T.rename(columns={0: "value"})

`m.metadata()` returns the **whole** settings tree as nested dicts. None values and lists are converted back from the way AcquireMP stores them. Here is a helper to print it as a tree:

In [ ]:
def print_tree(d, indent=0, max_depth=3):
    for k, v in d.items():
        if isinstance(v, dict):
            print("  " * indent + f"{k}/")
            if indent + 1 < max_depth:
                print_tree(v, indent + 1, max_depth)
        else:
            s = repr(v)
            print("  " * indent + f"{k} = {s[:70] + '…' if len(s) > 70 else s}")

print_tree(m.metadata(), max_depth=2)

Anything the API doesn't cover is still available through the underlying `h5py.File`, `m.h5`. For example:

In [ ]:
m.h5["movie/configuration/aods/x"]["scan_frequency"][()], list(m.h5["analysis/scores"])

## 2. The raw movie

`m.frames()` decodes the full movie into memory (`n_frames × ny × nx`, uint16, about 50 MB here). Each stored frame is the sum of `frame_binning` camera exposures. `m.frame(i)` fetches a single frame quickly by starting from the nearest stored keyframe.

`m.verify()` checks the decoded movie against every keyframe in the file.

In [ ]:
movie = m.frames()
t = m.times()                       # seconds, one value per frame
print(movie.shape, movie.dtype, f"{movie.nbytes/1e6:.0f} MB", "| keyframes match:", m.verify())

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13, 3.2), gridspec_kw={"width_ratios": [1, 1, 1.3]})
axs[0].imshow(movie[0]); axs[0].set_title("Raw frame 0")
axs[1].imshow(movie.mean(0)); axs[1].set_title("Time-averaged frame")
for a in axs[:2]: a.axis("off")

axs[2].plot(t, movie.mean(axis=(1, 2)), color=C_DATA, lw=1)
axs[2].set(xlabel="Time (s)", ylabel="Mean counts", title="Mean frame intensity")
fig.tight_layout()

The raw frames look like an almost uniform illumination field. Single particles are invisible at this stage, because their signal is well under 1% of the background. That's why mass photometry works with **ratiometric** images.

### Frame-quality scores

AcquireMP stores quality scores for each frame. Their exact definitions aren't documented, so treat them as relative measures.

In [ ]:
scores = pd.DataFrame(m.scores(), index=pd.Index(t, name="time (s)"))
valid = scores.dropna(axis=1, how="all")
fig, axs = plt.subplots(1, len(valid.columns), figsize=(4 * len(valid.columns), 2.6))
for ax, col in zip(np.atleast_1d(axs), valid.columns):
    ax.plot(valid.index, valid[col], color=C_DATA, lw=1); ax.set(title=col, xlabel="Time (s)")
fig.tight_layout()
print("Columns with no data in this file:", [c for c in scores if c not in valid])

## 3. The ratiometric movie

For each frame *i*, `m.ratiometric()` computes

$$r_i = \frac{\langle F\rangle_{[i,\,i+n)}}{\langle F\rangle_{[i-n,\,i)}} - 1$$

so a particle landing at frame *i* shows up at index *i*, the same index AcquireMP stores for its events. The sign follows AcquireMP: **binding is negative**, unbinding is positive. The first *n* and last *n*−1 frames are NaN.

In [ ]:
r = m.ratiometric()                  # float32, same shape as the movie
ev = m.events()                      # AcquireMP's fitted events (next section)
print(r.shape, "n_avg =", m.analysis_params["n_avg"])

Here is a strong binding event. The panels show a few frames before and after it lands, with the fitted position circled:

In [ ]:
binders = np.nonzero((ev["contrast"] < -0.015) & (ev["nn_distance"] > 10))[0]
k = binders[len(binders) // 2]
f0, x0, y0 = int(ev["frame"][k]), ev["x"][k], ev["y"][k]
offsets = [-6, -3, -1, 0, 1, 3, 6]
lim = 0.02

fig, axs = plt.subplots(1, len(offsets), figsize=(14, 2.4))
for ax, o in zip(axs, offsets):
    ax.imshow(r[f0 + o], vmin=-lim, vmax=lim)
    ax.add_patch(plt.Circle((x0, y0), 5, fill=False, color=C_ACCENT, lw=1.2))
    ax.set_xlim(x0 - 25, x0 + 25); ax.set_ylim(y0 + 25, y0 - 25)
    ax.set_title(f"frame {f0}" if o == 0 else f"{o:+d}"); ax.axis("off")
fig.suptitle(f"Event {k}: fitted contrast {ev['contrast'][k]:.4f}", y=1.04)

# contrast at the event's pixel over time
fig, ax = plt.subplots(figsize=(8, 2.6))
win = np.arange(f0 - 25, f0 + 25)
ax.plot(win - f0, r[win, int(round(y0)), int(round(x0))], color=C_DATA, marker="o", ms=3, lw=1)
ax.axhline(ev["contrast"][k], color=C_FIT, ls=":", lw=1, label="AcquireMP fitted contrast")
ax.set(xlabel="Frames relative to landing", ylabel="Ratiometric value", title="Centre-pixel trace"); ax.legend(frameon=False);

The trace rises and falls in a triangle about 2·*n* frames wide, which is what the sliding ratio produces from a step in the raw signal. It peaks at the frame AcquireMP recorded. The centre-pixel value is about 20% lower than AcquireMP's contrast, because AcquireMP fits the whole spot instead of reading one pixel.

### Browse the movie interactively

The slider needs `ipywidgets`, which the env includes. If the slider doesn't appear, the cell falls back to showing a static frame.

In [ ]:
def show_frame(i=int(np.bincount(ev["frame"].astype(int)).argmax()), lim=0.01):
    fig, ax = plt.subplots(figsize=(9, 3.8))
    ax.imshow(r[i], vmin=-lim, vmax=lim)
    sel = ev["frame"].astype(int) == i
    ax.scatter(ev["x"][sel], ev["y"][sel], s=90, facecolors="none", edgecolors=C_ACCENT, lw=1.2)
    ax.set_title(f"Ratiometric frame {i}  (t = {t[i]:.2f} s): {sel.sum()} fitted events")
    ax.axis("off"); plt.show()

try:
    from ipywidgets import interact, IntSlider, FloatSlider
    n = m.analysis_params["n_avg"]
    interact(show_frame, i=IntSlider(value=55, min=n, max=m.n_frames - n, step=1, continuous_update=False),
             lim=FloatSlider(value=0.01, min=0.002, max=0.05, step=0.002, continuous_update=False, readout_format=".3f"))
except ImportError:
    show_frame()

## 4. The fitted events from AcquireMP

`m.events()` returns AcquireMP's fitted events as numpy arrays. By default it keeps only events flagged *good* and *selected*. `m.events_df()` returns the same data as a pandas table. Pass `calibration=` to add a `mass` column (see section 6).

In [ ]:
df = m.events_df()
df["kind"] = np.where(df.contrast < 0, "binding", "unbinding")
display(df.head())
df.describe().T[["mean", "std", "min", "50%", "max"]]

In [ ]:
df.kind.value_counts()

### Fit quality

`fit_error` measures how well each spot matches the PSF model. You can apply a stricter cut than AcquireMP did. Its own limit, `max_error`, is in `analysis_params` above.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.2))
axs[0].hist(df.fit_error, bins=80, color=C_DATA)
axs[0].set(xlabel="fit_error", ylabel="Events", title="Distribution of fit errors")
axs[1].scatter(df.contrast, df.fit_error, s=3, alpha=0.4, color=C_DATA, lw=0)
axs[1].set(xlabel="Contrast", ylabel="fit_error", xlim=(-0.04, 0.03), title="Fit error vs contrast")
fig.tight_layout()

## 5. Contrast histogram and the Gaussian fits stored in the file

`m.gaussian_fits()` returns the peaks the operator fitted in AcquireMP's histogram view.

In [ ]:
fits = pd.DataFrame(m.gaussian_fits())
display(fits)

def gauss(x, a, mu, s):
    return a * np.exp(-0.5 * ((x - mu) / s) ** 2)

BW = 2e-4
edges = np.arange(-0.04, 0.03 + BW, BW)
counts, _ = np.histogram(df.contrast, bins=edges)
xc = 0.5 * (edges[1:] + edges[:-1])

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.stairs(counts, edges, fill=True, color=C_DATA, alpha=0.85, label="events")
xx = np.linspace(edges[0], edges[-1], 3000)
for _, p in fits.iterrows():
    # stored 'counts' = number of events under the peak -> convert to amplitude per bin
    amp = p.counts * BW / (p.sigma * np.sqrt(2 * np.pi))
    ax.plot(xx, gauss(xx, amp, p.position, p.sigma), color=C_FIT, lw=1)
ax.plot([], [], color=C_FIT, label="AcquireMP Gaussian fits")
ax.set(xlabel="Contrast", ylabel=f"Events / {BW:g} bin", title=f"{m.sample_info['sample']}: contrast histogram")
ax.legend(frameon=False);

The negative side is a clean **ladder** of evenly spaced binding peaks. The positive side mirrors its first few steps; those are particles *unbinding* from the glass.

## 6. Extra analyses

### 6a. Fitting the peaks yourself

Instead of relying on the GUI fits, find the binding peaks automatically and fit a sum of Gaussians with `scipy`. The fit includes one broad Gaussian for the background of unresolved events underneath the ladder. This makes the analysis reproducible across files.

In [ ]:
def multi_gauss(x, *p):
    return sum(gauss(x, *p[i:i + 3]) for i in range(0, len(p), 3))

def fit_peaks(contrasts, lo=-0.035, hi=-0.0015, bw=1e-4, smooth=3, prominence=3):
    # Find binding peaks in the histogram and fit them all at once as narrow Gaussians
    # on top of one broad background Gaussian (the unresolved events underneath the ladder).
    e = np.arange(lo, hi + bw, bw)
    c, _ = np.histogram(contrasts, bins=e)
    x = 0.5 * (e[1:] + e[:-1])
    cs = gaussian_filter1d(c.astype(float), smooth)
    idx, _ = find_peaks(cs, prominence=prominence)
    # background first, then one (amplitude, position, sigma) triplet per peak, each kept near its guess
    p0, lower, upper = [cs.min() + 1, x[idx].mean(), 0.01], [0, lo, 2e-3], [np.inf, hi, 0.05]
    for i in idx:
        p0 += [cs[i], x[i], 5e-4]
        lower += [0, x[i] - 1e-3, 1e-4]
        upper += [np.inf, x[i] + 1e-3, 1.5e-3]
    popt, pcov = curve_fit(multi_gauss, x, c, p0=p0, bounds=(lower, upper), maxfev=50000)
    perr = np.sqrt(np.diag(pcov))
    pk, pe = popt[3:], perr[3:]
    res = pd.DataFrame({"position": pk[1::3], "position_err": pe[1::3],
                        "sigma": pk[2::3], "amplitude": pk[0::3]})
    res["n_events"] = res.amplitude * res.sigma * np.sqrt(2 * np.pi) / bw
    return res.sort_values("position", ascending=False, ignore_index=True), (x, c, popt)

peaks, (x_h, c_h, popt) = fit_peaks(df.contrast)

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.stairs(c_h, np.append(x_h - 5e-5, x_h[-1] + 5e-5), fill=True, color=C_DATA, alpha=0.85, label="events")
xx = np.linspace(x_h[0], x_h[-1], 3000)
ax.plot(xx, multi_gauss(xx, *popt), color=C_FIT, lw=1.2, label=f"sum of {len(peaks)} Gaussians")
ax.plot(xx, gauss(xx, *popt[:3]), color=C_FIT, lw=1, ls=":", label="broad background")
ax.set(xlabel="Contrast", ylabel="Events / 1e-4 bin", title="Binding peaks, automatic fit"); ax.legend(frameon=False);
peaks.round(6)

### 6b. Is the ladder linear?

In mass photometry, contrast scales linearly with mass. If the peaks are *n*-mers of a single building block, plotting peak position against *n* should give a straight line through about zero. The slope is the contrast of one subunit.

In [ ]:
step0 = np.median(np.diff(np.sort(peaks.position.values)[::-1]))       # rough spacing (negative)
peaks["n"] = np.round(peaks.position / step0).astype(int)                # oligomer number assigned to each peak

slope, intercept = np.polyfit(peaks.n, peaks.position, 1, w=1 / peaks.position_err)
resid = peaks.position - (slope * peaks.n + intercept)

fig, axs = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1.4, 1]})
nn = np.arange(0, peaks.n.max() + 1)
axs[0].plot(nn, slope * nn + intercept, color=C_FIT, lw=1, label=f"fit: {slope:.5f}·n {intercept:+.5f}")
axs[0].errorbar(peaks.n, peaks.position, yerr=peaks.position_err, fmt="o", color=C_DATA, ms=6)
axs[0].set(xlabel="Assigned oligomer number n", ylabel="Peak contrast", title="Ladder linearity"); axs[0].legend(frameon=False)
axs[1].bar(peaks.n, resid * 1e4, color=C_DATA, width=0.6)
axs[1].axhline(0, color=C_FIT, lw=0.8)
axs[1].set(xlabel="n", ylabel="Residual (×10⁻⁴)", title="Residuals")
fig.tight_layout()
print(f"Contrast per subunit: {slope:.5f}   intercept: {intercept:+.5f}   (peaks assigned n = {list(peaks.n)})")

**Check the assigned *n* values.** They come from rounding each peak position to the typical spacing. If a step is missing in the ladder, the automatic assignment can be off by one, so correct `peaks["n"]` by hand if needed.

### 6c. Mass calibration

This ladder is Refeyn's **MassFerence P1** calibrant: oligomers of an 86 kDa protein, with certified peaks at 86, 172, 258 and 344 kDa. `calibrate()` finds the peaks, matches them to these masses and fits the line. The higher ladder peaks (430, 516, 602 kDa) are not used for the fit; they serve as an independent check of linearity.

For another calibrant, pass its masses: `calibrate(file, [66, 132, 480])`.

In [ ]:
from mprfile import calibrate
cal = calibrate(DATA, "MassFerence P1")
print(cal.report())

# the same result by hand, from the ladder fit in 6b: n-mer = n × 86 kDa
manual = Calibration.fit(peaks.position, peaks.n * 86.0)
print("\nManual fit on all ladder steps:", manual)

In [ ]:
evc = m.events(calibration=cal)
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.hist(evc["mass"], bins=np.arange(-300, 800, 5), color=C_DATA)
for k in range(1, 8):
    ax.axvline(86 * k, color=C_FIT, lw=0.8, ls=":")
ax.set(xlabel="Mass (kDa)", ylabel="Events per 5 kDa",
       title="Calibrant in mass units (dotted lines: n × 86 kDa; negative = unbinding)");

Section 7 uses this calibration to analyse a sample.

### 6d. Landing kinetics

The number of landings per second shows how fast particles reach the surface. Its decline over time shows the solution depleting or the glass surface filling up. Here binding and unbinding are counted separately.

In [ ]:
bins = np.arange(0, t[-1] + 1, 1.0)
fig, axs = plt.subplots(1, 2, figsize=(12, 3.4))
for kind, color in [("binding", C_DATA), ("unbinding", C_ACCENT)]:
    tt = df.time[df.kind == kind]
    rate, _ = np.histogram(tt, bins=bins)
    axs[0].stairs(rate, bins, color=color, lw=1.4, label=kind)
    axs[1].plot(np.sort(tt), np.arange(1, len(tt) + 1), color=color, lw=1.4, label=kind)
axs[0].set(xlabel="Time (s)", ylabel="Events / s", title="Event rate"); axs[0].legend(frameon=False)
axs[1].set(xlabel="Time (s)", ylabel="Cumulative events", title="Cumulative events"); axs[1].legend(frameon=False)

# single-exponential fit to cumulative binding: N(t) = N_inf (1 - exp(-t/tau))
tb = np.sort(df.time[df.kind == "binding"]); Nb = np.arange(1, len(tb) + 1)
(Ninf, tau), _ = curve_fit(lambda x, a, b: a * (1 - np.exp(-x / b)), tb, Nb, p0=[Nb[-1] * 2, t[-1]])
axs[1].plot(tb, Ninf * (1 - np.exp(-tb / tau)), color=C_FIT, ls="--", lw=1, label="exp. fit")
axs[1].legend(frameon=False)
fig.tight_layout()
print(f"Initial landing rate ≈ {Ninf / tau:.0f} /s, time constant τ ≈ {tau:.0f} s")

### 6e. Spatial distribution and contrast drift

Landing positions should cover the field of view evenly. If the peak positions drift over the minute of recording, that points to a focus or illumination change.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 3.4), gridspec_kw={"width_ratios": [1.5, 1]})
ny, nx = m.frame_shape
h = axs[0].hist2d(df.x, df.y, bins=[np.arange(0, nx + 5, 5), np.arange(0, ny + 5, 5)], cmap="Blues")
axs[0].invert_yaxis(); axs[0].set_aspect("equal"); axs[0].grid(False)
axs[0].set(xlabel="x (px)", ylabel="y (px)", title="Landing positions")
fig.colorbar(h[3], ax=axs[0], fraction=0.02, label="events / 5×5 px")

# drift: median contrast of the three strongest-populated binding peaks, per 10 s window
windows = np.arange(0, t[-1] + 10, 10)
for (_, p), col in zip(peaks.nlargest(3, "n_events").sort_values("n").iterrows(), [C_DATA, C_ACCENT, C_FIT]):
    sel = (df.contrast - p.position).abs() < 1.5 * p.sigma
    med = [df.contrast[sel & (df.time >= a) & (df.time < b)].median() for a, b in zip(windows[:-1], windows[1:])]
    axs[1].plot(windows[:-1] + 5, (np.array(med) / p.position - 1) * 100, marker="o", ms=4, lw=1.2, color=col, label=f"n = {int(p.n)}")
axs[1].axhline(0, color=C_FIT, lw=0.8)
axs[1].set(xlabel="Time (s)", ylabel="Peak shift (%)", title="Contrast drift per 10 s"); axs[1].legend(frameon=False)
fig.tight_layout()

### 6f. The average point-spread function

This crops the ratiometric frame around every well-isolated binding event, normalises each crop by its fitted contrast, and averages them. The result is the instrument's interferometric PSF: a dark core with a bright ring. A lopsided PSF would mean the focus or alignment was off.

In [ ]:
HALF = 10
good = (df.kind == "binding") & (df.nn_distance > 2 * HALF) & (df.x > HALF) & (df.x < nx - HALF - 1) \
       & (df.y > HALF) & (df.y < ny - HALF - 1) & (df.contrast < -0.004)
stack = []
for _, e in df[good].iterrows():
    xi, yi = int(round(e.x)), int(round(e.y))
    crop = r[int(e.frame), yi - HALF:yi + HALF + 1, xi - HALF:xi + HALF + 1]
    if np.isfinite(crop).all():
        stack.append(crop / -e.contrast)
psf = np.mean(stack, axis=0)

yy, xx_ = np.indices(psf.shape) - HALF
rad = np.hypot(xx_, yy)
rb = np.arange(0, HALF)
prof = [psf[(rad >= a) & (rad < a + 1)].mean() for a in rb]

fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
lim = np.abs(psf).max()
im = axs[0].imshow(psf, vmin=-lim, vmax=lim, extent=[-HALF - .5, HALF + .5, HALF + .5, -HALF - .5])
axs[0].set(title=f"Mean PSF ({len(stack)} events)", xlabel="px", ylabel="px"); axs[0].grid(False)
fig.colorbar(im, ax=axs[0], fraction=0.045, label="normalised contrast")
axs[1].plot(rb + 0.5, prof, color=C_DATA, marker="o", ms=4, lw=1.2)
axs[1].axhline(0, color=C_FIT, lw=0.8)
axs[1].set(xlabel="Radius (px)", ylabel="Normalised contrast", title="Radial profile")
fig.tight_layout()

## 7. From contrast to mass: analysing a sample

Everything so far used the calibrant. Now a **sample**. `analyze_sample()` converts AcquireMP's particles to mass with the calibration from 6c, finds the peaks and fits them, and runs the quality checks. This is exactly what `analyze_measurements.ipynb` and the `mpr-analyze` command do for each sample.

In [ ]:
from mprfile import analyze_sample, AnalysisSettings
from mprfile.report import plot_sample, plot_calibration

SAMPLE = Path("019_252_50nM.mpr")      # <-- a sample measured with the same settings as the calibrant

res_auto = analyze_sample(SAMPLE, cal)
display(res_auto.peaks)                # mass, ± fit, σ, events, % and notes per peak
res_auto.qc                            # event counts, landing rate, share assigned to peaks

In [ ]:
print("\n".join(res_auto.warnings) or "no warnings")
res_auto.events.head()                 # every particle: contrast, mass (kDa), x, y, frame, time, fit errors ...

The report page, as saved to `report.pdf`:

In [ ]:
fig = plot_sample(res_auto); plt.show()

### Settings

`AnalysisSettings` holds every parameter of the automatic analysis (the docstring explains each: `help(AnalysisSettings)`). Changing one is a one-liner, for example a narrower bin width and dropping poorly fitted particles:

In [ ]:
from dataclasses import asdict
display(pd.Series(asdict(AnalysisSettings()), name="default").to_frame())

res_strict = analyze_sample(SAMPLE, cal, AnalysisSettings(bin_width=4, max_fit_error=0.2))
res_strict.peaks[["mass_kDa", "mass_err_kDa", "sigma_kDa", "counts", "percent", "notes"]].round(1)

### The peak finder underneath

The automatic mode uses `mprfile.peaks.fit_peaks`: it finds peaks on a lightly smoothed histogram, then fits all of them at once as Gaussians. It works on any values, including contrasts.

In [ ]:
from mprfile.peaks import fit_peaks
binding_mass = res_auto.events.mass[res_auto.events.mass > 0]
pf = fit_peaks(binding_mass, 40, 1500, 5, sigma_max=40)
pf.peaks.round(2), pf.notes

## 8. Calibrations: other calibrants, checks and reuse

- `CALIBRANTS` lists the built-in calibrants. Any other calibrant works by passing its masses in kDa.
- The calibration checks itself: if the masses don't fit the peaks, it says so. Here, deliberately **wrong** masses are passed for the ladder.
- `resolution_kDa` is the typical width of a single species (from the calibrant peaks). The ROI checks below use it.
- A calibration can be saved and reused for later samples.

In [ ]:
from mprfile import CALIBRANTS
print(CALIBRANTS["MassFerence P1"]["masses"], "kDa")

wrong = calibrate(DATA, [66, 132, 480])          # wrong calibrant masses for this ladder
print("\n".join(wrong.warnings))

print(f"\nresolution σ ≈ {cal.resolution_kDa:.1f} kDa · calibrated range {cal.mass_range} kDa")
cal.save("my_calibration.json")
cal_again = Calibration.load("my_calibration.json")
print(cal_again)

In [ ]:
fig = plot_calibration(cal); plt.show()

## 9. Choosing regions and the number of peaks yourself: overfitting statistics

Sometimes you know more than the automatic peak finder, for example that a shoulder is a real species. You can then define a **region of interest (ROI)** and choose how many Gaussians to fit in it. The danger is that more Gaussians *always* fit better, even when the extra ones are noise.

`mprfile.mixture.fit_roi` therefore:
1. fits the chosen number `k` by **maximum likelihood on the individual masses** (no histogram bins), with a flat background for the events between peaks;
2. fits the same ROI with 1…4 Gaussians and compares them with **BIC** (lower is better; ΔBIC > 2 / 6 / 10 = positive / strong / very strong evidence against a model);
3. checks each component: resolved from its neighbours (Ashman's D ≥ 2), plausible width relative to the instrument resolution, enough events, not at the ROI edge, parameters not stuck at a limit;
4. optionally refits resampled data (**bootstrap**) to see whether the components are stable.

In [ ]:
from mprfile.mixture import fit_roi, bootstrap_roi

roi = fit_roi(binding_mass, 400, 620, k=2, resolution=cal.resolution_kDa)
print(roi.model_summary(), "| risk:", roi.risk)
display(roi.fit.table().round(2))          # the fitted components
display(roi.models.round(1))               # 1…4 Gaussians compared (log L, AIC, BIC, ΔBIC)
roi.warnings

What happens when you ask for **too many** (or too few) Gaussians in the same region:

In [ ]:
edges = np.arange(400, 625, 5); xc = 0.5 * (edges[1:] + edges[:-1])
h, _ = np.histogram(binding_mass, edges)
fig, axs = plt.subplots(1, 3, figsize=(14, 3.4), sharey=True)
for ax, k in zip(axs, [1, 2, 4]):
    rk = fit_roi(binding_mass, 400, 620, k=k, resolution=cal.resolution_kDa, models=roi.models)
    ax.stairs(h, edges, fill=True, color=C_DATA, alpha=0.8)
    xx = np.linspace(400, 620, 600)
    comps, bg = rk.fit.density(xx, 5, per_component=True)
    for c in comps:
        ax.plot(xx, c, color=C_FIT, lw=1)
    ax.plot(xx, rk.fit.density(xx, 5), color=C_ACCENT, lw=1.4)
    ax.set(title=f"k = {k}: {rk.risk} risk", xlabel="Mass (kDa)")
    print(f"k={k}: " + ("; ".join(rk.warnings) if rk.warnings else "no warnings"))
axs[0].set_ylabel("Events per 5 kDa"); fig.tight_layout()

With k = 1 the 459 kDa species is missed (*underfitting*). k = 2 is what BIC prefers. Note that **k = 4 looks just as good** by eye: the two extra broad components quietly absorb the background. That's exactly why the statistics are needed. They flag the extra components as not supported (*overfitting*): ΔBIC +22, unresolved neighbours, poorly determined positions.

A **bootstrap** check refits 50 resampled data sets. Real peaks keep their position; noise peaks don't.

In [ ]:
boot = bootstrap_roi(binding_mass, roi, n_boot=50, resolution=cal.resolution_kDa)
boot.round(2)

### ROIs in the regular analysis

Put the ROIs in the settings as `(lo, hi, k)`. The result has the same shape as the automatic analysis, plus the per-ROI statistics (`res_roi.rois`) and `roi_*` columns in the peak table. The report page shows the ROIs and a model check.

In [ ]:
res_roi = analyze_sample(SAMPLE, cal, AnalysisSettings(rois=[(400, 620, 2), (40, 160, 1)]))
display(res_roi.peaks[["mass_kDa", "mass_err_kDa", "sigma_kDa", "counts", "percent", "roi", "roi_risk", "notes"]].round(1))
fig = plot_sample(res_roi); plt.show()

## 10. Many files at once, and the command line

`analyze()` calibrates once and analyses a whole set of files. It writes everything (reports, CSV tables, `calibration.json`, `summary.csv`) to an output folder. The calibrant file is found automatically when its name contains *ladder*, *calib*, *massference* or *standard*, or you pass `calibrant_file=`.

In [ ]:
from mprfile import analyze, find_calibrant_file
print("calibrant detected:", find_calibrant_file(sorted(Path(".").glob("*.mpr"))))

batch = analyze("*.mpr", out="results_tutorial", verbose=False)       # settings=... works here too
display(batch.summary[["sample", "file", "peak", "mass_kDa", "sigma_kDa", "percent", "notes"]].round(1))
sorted(str(p.relative_to("results_tutorial")) for p in Path("results_tutorial").rglob("*") if p.is_file())

The same from a terminal (Anaconda Prompt, `mpr` env active) is **`mpr-analyze`**. The cell below calls it through Python, so it works regardless of your PATH. Common forms:

```
mpr-analyze                                        # all *.mpr here, auto-detected calibrant
mpr-analyze data\*.mpr -c data\002_Ladder.mpr -o results
mpr-analyze new\*.mpr --calibration results\calibration\calibration.json
mpr-analyze --roi 400 620 2 --roi 40 160 1 --bootstrap 100
```

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, "-m", "mprfile.cli", "--help"], capture_output=True, text=True).stdout)

## 11. Export

- **Events to CSV**, for Excel, Origin or R. Pass `calibration=cal` to include masses.
- **Movie to TIFF**, for Fiji/ImageJ. This needs `tifffile`, which is in the env. The files are large (about 50 MB raw, about 100 MB ratiometric), so these lines are commented out.

In [ ]:
out = Path("exports"); out.mkdir(exist_ok=True)
m.export_events_csv(out / f"{DATA.stem}_events.csv", calibration=cal)
peaks.to_csv(out / f"{DATA.stem}_peaks.csv", index=False)
# m.export_movie_tiff(out / f"{DATA.stem}_raw.tif")
# m.export_movie_tiff(out / f"{DATA.stem}_ratiometric.tif", ratiometric=True)
sorted(p.name for p in out.iterdir())

## 12. The interactive ROI tool

`ROIFitter` puts section 9 into a point-and-click tool.

- **Drag across the histogram** to draw an ROI. The *Gaussians* dropdown next to *Add ROI* sets how many are fitted: *BIC best* or a number. You can also type the limits.
- Each ROI gets a row where you change the number of Gaussians, jump to the BIC-preferred number, or delete it. The plot, the residual panel (bars beyond ±2σ are orange) and the statistics update immediately.
- **Bootstrap check** runs the stability test, and **Save** writes the report, `peaks.csv`, `rois.json` and `roi_model_comparison.csv`, and updates `summary.csv`.
- The line under the tool reproduces your choice as code or as an `mpr-analyze` command.

Mouse drawing needs `ipympl` (in the env; restart JupyterLab after installing it). Without it, the tool still works with typed limits. This section is last because `%matplotlib widget` switches all later plots to interactive mode.

In [ ]:
try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    print("Mouse drawing is not available (ipympl missing or JupyterLab not restarted). Typed limits still work.")
from mprfile.interactive import ROIFitter

fitter = ROIFitter(batch)            # or ROIFitter(res_auto) for a single sample
fitter

Everything the buttons do is also available in code, handy for scripting or for checking a choice made by someone else:

In [ ]:
fitter.add_roi(400, 620)             # k=None -> BIC-preferred number of Gaussians
fitter.add_roi(40, 160, k=2)         # deliberately too many: see the HIGH RISK label and the checks
fitter.set_k(0, 1)                   # ROIs are sorted by mass: index 0 is 40-160 kDa
print([(r.label, r.k, r.risk) for r in fitter.rois])
print(fitter.settings().rois)        # -> AnalysisSettings(rois=...) for reuse with analyze()
# fitter.bootstrap(100); fitter.save("results_tutorial")

In [ ]:
m.close()

---
**Where to look next**
- `help(MPRFile)`, `help(calibrate)`, `help(AnalysisSettings)`, `help(fit_roi)`: every option, documented.
- `README.md`: the file format and the statistics in more detail.

**Notes:** the format was reverse-engineered from AcquireMP 2025.1.2 files (file format v4).